#### 1- Initilazation

Import the libraries used in the completed notebook and select the available PyTorch device. This keeps the setup in one place before working with the dataset.

In [6]:
import random

import pandas as pd
import torch

from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


#### Hyperparametes


In [1]:
seed = 42
#Etc

#### 2- Configs

Store the dataset and model identifiers in one place so later cells can use consistent names. The model is only identified here and is not loaded in this section.

In [7]:
# Dataset and model configuration

DATASET_NAME = "ai4privacy/pii-masking-openpii-1.5m"
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"

#### 3- Loading Dataset

Load the configured Hugging Face dataset and its available splits. This step is kept separate because the dataset is large and may take time to download.

In [8]:
dataset = load_dataset(DATASET_NAME)

dataset

Generating validation split: 100%|██████████| 326463/326463 [00:22<00:00, 14294.03 examples/s]


DatasetDict({
    train: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes', 'source_dataset'],
        num_rows: 1309912
    })
    validation: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes', 'source_dataset'],
        num_rows: 326463
    })
})

### 4- Dataset Citation and License

This notebook uses the **OpenPII 1.5M: Multilingual PII Masking Dataset with Asia Pacific Extension**, published by **Ai4Privacy / Ai Suisse SA** on Hugging Face.

The dataset contains synthetic personally identifiable information (PII) and is released under the **CC BY 4.0** license. It may be used for research, commercial use, redistribution, and modification, provided appropriate attribution is given.

**Dataset:** `ai4privacy/pii-masking-openpii-1.5m`  
**Publisher:** Ai4Privacy / Ai Suisse SA  
**License:** CC BY 4.0  
**Year:** 2026  
**Source:** https://huggingface.co/datasets/ai4privacy/pii-masking-openpii-1.5m

#### BibTeX

```bibtex
@dataset{ai4privacy_openpii_1_5m_2026,
  author = {Ai4Privacy},
  title = {OpenPII 1.5M: Multilingual PII Masking Dataset with Asia Pacific Extension},
  year = {2026},
  publisher = {Hugging Face},
  url = {https://huggingface.co/datasets/ai4privacy/pii-masking-openpii-1.5m}
}
```

#### 5- Get English text only

Filter every dataset split to examples marked as English. This keeps all English rows, including examples whose `privacy_mask` is empty.

In [9]:
english_dataset = dataset.filter(
    lambda example: example["language"] == "en"
)

print("Original dataset:")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data):,}")

print("\nEnglish-only dataset:")
for split_name, split_data in english_dataset.items():
    print(f"  {split_name}: {len(split_data):,}")

Filter: 100%|██████████| 326463/326463 [04:04<00:00, 1336.04 examples/s]

Original dataset:
  train: 1,309,912
  validation: 326,463

English-only dataset:
  train: 163,740
  validation: 40,909


#### 6. Show five random examples

Select up to five random English training examples and display each source text beside its masked text. Rerunning the next cell draws another random sample.

In [10]:
sample_size = min(5, len(english_dataset["train"]))

sample_indices = random.sample(
    range(len(english_dataset["train"])),
    k=sample_size,
)

random_examples = english_dataset["train"].select(sample_indices)

examples_df = pd.DataFrame({
    "Source text": random_examples["source_text"],
    "Masked text": random_examples["masked_text"],
})

pd.set_option("display.max_colwidth", None)

examples_df

,Source text,Masked text
0,"The fund committee, chaired by Gov Naike Meregalli, will review your progress report, which should include your current 11 and Female statistics, as well as a brief overview of your branding strategy impact. Submit the report to CP@protonmail.com no later than 27/04/1979, and ensure your billing address uses 453001 for accurate processing.","The fund committee, chaired by [TITLE_1] [GIVENNAME_1] [SURNAME_1], will review your progress report, which should include your current [AGE_1] and [SEX_1] statistics, as well as a brief overview of your branding strategy impact. Submit the report to [EMAIL_1] no later than [DATE_1], and ensure your billing address uses [ZIPCODE_1] for accurate processing."
1,"Norea Jeannett: 'Hey, can you send the schematics for the antique brass chandelier?'\nUmaro Sureshbhai Moberg: 'Sure, I’ll email them to T@outlook.com shortly.'","[GIVENNAME_2]: 'Hey, can you send the schematics for the antique brass chandelier?'\n[GIVENNAME_1] [SURNAME_1]: 'Sure, I’ll email them to [EMAIL_1] shortly.'"
2,"Racer Yeremy Yerga Gusse, aged 44 and identified by 17185723093, will be assigned to the western dunes sector near East Wildcat Way 3509 in Edwards (ZIP 27925); a quick check of their 257-36-5148 and 71936 21691 will be performed on arrival to ensure full compliance.","Racer [GIVENNAME_1] [SURNAME_1], aged [AGE_1] and identified by [IDCARDNUM_1], will be assigned to the western dunes sector near [STREET_1] [BUILDINGNUM_1] in [CITY_1] (ZIP [ZIPCODE_1]); a quick check of their [SOCIALNUM_1] and [TAXNUM_1] will be performed on arrival to ensure full compliance."
3,"Elisabelle (09:15): ""Hey team, does anyone have tips for reducing glare when using the new Gen‑III goggles?""\nDeshi Mejdina (09:17): ""I found that wearing a thin, neutral‑colored visor and allowing 20‑30 minutes of dark adaptation before entering the field really helps.""\nPerka (09:20): ""Good call, I’ll add that to the briefing deck.""","[GIVENNAME_3] (09:15): ""Hey team, does anyone have tips for reducing glare when using the new Gen‑III goggles?""\n[GIVENNAME_2] (09:17): ""I found that wearing a thin, neutral‑colored visor and allowing 20‑30 minutes of dark adaptation before entering the field really helps.""\n[GIVENNAME_1] (09:20): ""Good call, I’ll add that to the briefing deck."""
4,"The final stunt schedule, approved by Mr Loriana Dominoni Reuteler Gammarano (age 35), will be archived under tax reference 39786-27988 and secured with a credit‑card verification code 3588489162587660 for compliance audit.","The final stunt schedule, approved by [TITLE_1] [GIVENNAME_1] [SURNAME_1] (age [AGE_1]), will be archived under tax reference [TAXNUM_1] and secured with a credit‑card verification code [CREDITCARDNUMBER_1] for compliance audit."
